<a href="https://colab.research.google.com/github/mhmmdrdhiansyah/data-science-2026/blob/main/pertemuan3_muhammad_ardhiansyah_240401020092.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

nama : Muhammad ardhianyshah
<br>
nim : 240401020092
<br>
kelas : IF403

In [21]:
import pandas as pd
import numpy as np
import requests
import json
from scipy.stats.mstats import winsorize

# Perbaikan Path: Menggunakan jalur absolut Google Colab agar tidak error
path_file = '/content/housing_dirty.csv'
df = pd.read_csv(path_file)

print('--- STEP 0: Eksplorasi Awal ---')
print('Dimensi awal dataset:', df.shape)
print('\nJumlah missing values per kolom:')
print(df.isnull().sum())
display(df.head())

--- STEP 0: Eksplorasi Awal ---
Dimensi awal dataset: (130, 7)

Jumlah missing values per kolom:
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,Bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang


In [ ]:
print('--- PEMBERSIHAN DATA ---')

# STEP 1 — Hapus Duplikat
df.drop_duplicates(inplace=True)
print(f'Setelah hapus duplikat: {df.shape}')

# STEP 2 — Normalisasi Format String
if 'kota' in df.columns:
    df['kota'] = df['kota'].str.strip().str.title()
if 'kondisi' in df.columns:
    df['kondisi'] = df['kondisi'].str.strip().str.lower()
print('Normalisasi string selesai.')

# STEP 3 — Imputasi Missing Values
# Imputasi numerik kontinu dengan Median
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())

# Antisipasi kolom tahun_bangun (diimputasi sebelum masuk perhitungan outlier)
if 'tahun_bangun' in df.columns:
    df['tahun_bangun'] = df['tahun_bangun'].fillna(df['tahun_bangun'].median())

# Imputasi numerik diskrit dengan Modus
if 'kamar' in df.columns and not df['kamar'].mode().empty:
    df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

print('Imputasi missing values selesai.')

In [23]:
print('--- PENANGANAN OUTLIER & EKSPOR ---')

# STEP 4 — Tangani Outlier (IQR Fence)
# Menyesuaikan kolom secara dinamis berdasarkan kolom yang benar-benar ada di dataset Anda
kolom_target = [col for col in ['harga_juta', 'luas_m2', 'tahun_bangun'] if col in df.columns]

for col in kolom_target:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    pagar_bawah = Q1 - 1.5 * IQR
    pagar_atas = Q3 + 1.5 * IQR

    # Membatasi nilai ekstrem (Capping/Clipping)
    df[col] = df[col].clip(pagar_bawah, pagar_atas)
print('Penanganan outlier dengan IQR Fence selesai.')

# STEP 5 — Validasi Kualitas Data & Ekspor Berkas Bersih
assert df.isnull().sum().sum() == 0, 'Peringatan: Masih ada missing values!'
assert df.duplicated().sum() == 0, 'Peringatan: Masih ada data duplikat!'

print('\n[VALIDASI SUKSES] Kualitas data bersih terjamin!')
print('Shape akhir dataset bersih:', df.shape)

# Simpan hasil pembersihan
df.to_csv('/content/housing_clean.csv', index=False)
print("File 'housing_clean.csv' berhasil diekspor!")

--- PENANGANAN OUTLIER & EKSPOR ---
Penanganan outlier dengan IQR Fence selesai.

[VALIDASI SUKSES] Kualitas data bersih terjamin!
Shape akhir dataset bersih: (130, 7)
File 'housing_clean.csv' berhasil diekspor!


In [24]:
print('--- LANJUTAN: Ekstraksi Data REST API Publik ---')

# Menghubungi endpoint REST API simulasi menggunakan library requests
URL_API = "https://jsonplaceholder.typicode.com/users"
response = requests.get(URL_API)

print(f"Status Koneksi HTTP: {response.status_code} (200 = Sukses)")

if response.status_code == 200:
    data_json = response.json()

    # Mengurai nested JSON bertingkat menjadi flat DataFrame sesuai isi modul
    df_users = pd.json_normalize(data_json, sep='_')

    print(f"Berhasil mengekstrak {df_users.shape[0]} data pengguna dari API!")
    print("\nSampel Hasil Ekstraksi API Bivariat:")
    display(df_users[['id', 'name', 'email', 'address_city', 'company_name']].head(3))

--- LANJUTAN: Ekstraksi Data REST API Publik ---
Status Koneksi HTTP: 200 (200 = Sukses)
Berhasil mengekstrak 10 data pengguna dari API!

Sampel Hasil Ekstraksi API Bivariat:


,id,name,email,address_city,company_name
0,1,Leanne Graham,Sincere@april.biz,Gwenborough,Romaguera-Crona
1,2,Ervin Howell,Shanna@melissa.tv,Wisokyburgh,Deckow-Crist
2,3,Clementine Bauch,Nathan@yesenia.net,McKenziehaven,Romaguera-Jacobson


Pada Modul / Pertemuan 3 (Data Cleaning: Missing Values, Outlier & Ekstraksi Data) ini, saya mempelajari siklus krusial dalam menyiapkan data mentah agar menjadi informasi berkualitas yang siap diolah untuk kebutuhan pemodelan statistik dan Machine Learning. Melalui modul ini, saya memahami taksonomi masalah kualitas data, seperti pentingnya mengeliminasi baris ganda (duplikat), mendeteksi sebaran data kosong (missing values), serta menerapkan teknik imputasi yang tepat—yakni menggunakan nilai Median untuk menjaga konsistensi data numerik kontinu dari pengaruh skewness serta menggunakan Modus untuk data diskrit. Lebih jauh lagi, modul ini membekali saya dengan kemampuan matematis untuk menyaring dan membatasi data ekstrem menggunakan metode Interquartile Range (IQR Fence / Capping) serta melakukan standardisasi inkonsistensi string pada data kategorikal. Di akhir materi, saya juga mempelajari perluasan kompetensi teknis berupa teknik ekstraksi data dari sumber eksternal, baik melalui pembacaan data semi-terstruktur berupa file JSON lokal maupun melakukan pemanggilan langsung ke REST API Publik menggunakan library requests untuk kemudian diratakan menjadi bentuk tabel dua dimensi siap pakai menggunakan fungsi pd.json_normalize().